In [1]:
"""
Search for weight-6 qudit generalized toric codes on twisted tori
=================================================================

What this notebook does
-----------------------
For a prime q, a target number of logical qudits k and a list of block
lengths n, it looks for pairs of Laurent polynomials (f, g) and twisted tori
(alpha, beta, gamma) such that the code defined by

    A_v = (f, g, 0, 0),   B_p = (0, 0, gbar, -fbar)

on the torus with identifications  y^alpha = 1,  x^beta y^gamma = 1
has exactly k logical qudits.  The number of logical qudits is computed
from the quotient ring

    k = 2 * dim_Fq  Fq[x^±1, y^±1] / < f, g, y^alpha - 1, x^beta y^gamma - 1 >

with a Groebner basis.  Distances are not computed here; they are estimated
afterwards with the GAP script (QDistRnd) from the CSV files written below.

Pipeline
--------
  1. random_fg            draws a random pair (f, g) of the weight-6 form
                          f = 1 + l1 x^a1 y^b1 + l2 x^a2 y^b2,  g likewise
  2. laurent_coprime_fg   keeps only coprime pairs (topological-order condition)
  3. laurent_groebner_and_basis  gives k_max (no torus) and k on a torus
  4. search_alpha_beta_k_improve_FG  runs over all tori with 2*alpha*beta = n
                          and all twists gamma, keeps those with k = k_target,
                          and checks that the code is a valid CSS code with
                          full check weight
  5. search_random_fg     repeats 1-4 for num_fg random pairs and writes
                          n{n}fg_{k}k_{q}q_{w}wait.csv
  6. eliminate_minus      rewrites f, g without negative exponents (needed by
                          the GAP script) and writes minus_n{n}fg_...wait.csv
  7. run_for_n            does 5 and 6 for one n; called in parallel for a
                          list of n values

Output files (one folder per n:  data/data_{n}n_{k}k_{q}q_{w}w/)
  n{n}fg_...wait.csv      columns: f, g, n, alpha, beta, gamma, k_gamma0, k_gamma
                          (k_gamma0 = k on the untwisted torus, k_gamma = k on
                          the twisted one)
  minus_n{n}fg_...wait.csv  the same rows plus f_no_neg, g_no_neg (f and g
                          multiplied by a monomial so that all exponents are
                          >= 0) and the shifts used; this is the GAP input

Reference
---------
This code accompanies

    M. Halla, "Qudit Twisted-Torus Codes in the Bivariate Bicycle Framework",
    arXiv:2602.04443, https://arxiv.org/abs/2602.04443

Please cite this paper if you use the code or the codes found with it.
The qubit construction it extends is Z. Liang, K. Liu, H. Song and
Y.-A. Chen, PRX Quantum 6, 020357 (2025).

Software: SageMath 10.9.
Copyright (c) 2026 Mourad Halla.  Licence: MIT (see LICENSE).
"""

import csv
import os
import random
import re

##############################################
# Parameters (independent of n_target)
##############################################
q = 3               # local dimension of the qudits (prime); all algebra is over GF(q)

w = 6               # check weight of the ansatz: wt(f) + wt(g) = 3 + 3

k_target  = 4       # number of logical qudits we look for; also used to
                    # discard pairs (f, g) whose k_max is smaller than this

num_fg    = 1000    # number of random pairs (f, g) tried per n

max_exp   = 3       # exponents a, b, c, d of the monomials lie in [-max_exp, max_exp]

seed = 1024         # random seed, so that a run can be repeated exactly

##############################################
# Rings and basic setup (Laurent)
##############################################

Fq = GF(q)                                            # the field F_q
P.<x,y> = LaurentPolynomialRing(Fq, order='lex')      # R_q = F_q[x^±1, y^±1]

##############################################
# Antipode  f(x,y) -> f(x^-1, y^-1)
##############################################

def antipode(f):
    """The antipode map of the paper: every monomial x^a y^b becomes x^-a y^-b.
    It appears in the Z-type generator B_p = (0, 0, gbar, -fbar)."""
    return f(x^-1, y^-1)

##############################################
# Groebner + basis for Laurent ideals
##############################################

def laurent_groebner_and_basis(R, gens, order='lex'):
    """
    Dimension of the quotient of the Laurent polynomial ring R by the ideal
    generated by `gens`, computed with a Groebner basis.

    The computation is done in the ordinary polynomial ring S = F_q[x, y]:
    every generator is first multiplied by the smallest monomial that removes
    its negative exponents (this does not change the ideal in R, because
    monomials are units there).  This is done separately for each generator;
    a single common factor would also multiply the torus relations
    y^alpha - 1 and x^beta y^gamma - 1 and create spurious solutions with
    x = 0 or y = 0.  For the same reason the ideal is saturated with respect
    to x*y afterwards, which removes any remaining solution on the axes.

    Returns
        G_L   : a Groebner basis of the ideal in R (not used further)
        basis : the normal basis of the ideal in S, i.e. the monomials not
                divisible by any leading monomial of the Groebner basis.
                Its length is dim_Fq of the quotient; k is twice this length.
    """
    I_L = R.ideal(gens)
    G_L = I_L.groebner_basis()   # not used later, but returned

    n      = R.ngens()
    names  = R.variable_names()
    k      = R.base_ring()

    # auxiliary polynomial ring
    S = PolynomialRing(k, names, order=order)

    # clear negative exponents SEPARATELY for each generator.
    polys = []
    for f in gens:
        f    = R(f)
        mons = f.monomials()
        factor = R(1)
        for i in range(n):
            xi  = R.gen(i)
            m_i = min(m.degree(xi) for m in mons)
            if m_i < 0:
                factor *= xi**(-m_i)
        polys.append(S(f * factor))

    J = S.ideal(polys)

    # remove any solutions on the axes x = 0 or y = 0 (not in the Laurent ring)
    J = J.saturation(S.ideal(prod(S.gens())))[0]

    basis_poly    = J.normal_basis()
    basis_laurent = [R(b) for b in basis_poly]   # length is what we need

    return G_L, basis_laurent

##############################################
# Coprime check for Laurent f,g
##############################################

def laurent_coprime_fg(f, g, order='lex'):
    """
    True if f and g have no common factor, which is the topological-order
    condition <f> ∩ <g> = <f g> of the paper.

    Negative exponents are cleared with one common monomial and the gcd is
    taken in F_q[x, y].  Because f and g both have constant term 1, neither
    is divisible by x or y, so the gcd is a constant exactly when the two
    Laurent polynomials are coprime.
    """
    R = f.parent()
    k = R.base_ring()
    n = R.ngens()
    names = R.variable_names()

    all_mons = list(f.monomials()) + list(g.monomials())

    mins = []
    for i in range(n):
        xi = R.gen(i)
        mins.append(min(m.degree(xi) for m in all_mons))

    factor = R(1)
    for i in range(n):
        if mins[i] < 0:
            factor *= R.gen(i)**(-mins[i])

    S = PolynomialRing(k, names, order=order)
    f_poly = S(R(f) * factor)
    g_poly = S(R(g) * factor)

    gcd_fg = f_poly.gcd(g_poly)

    if gcd_fg == 0:
        return False
    return gcd_fg.degree() == 0

##############################################
# Twisted torus shift matrices X, Y
##############################################

def twisted_shifts(alpha, beta, gamma):
    """
    Permutation matrices X and Y that shift a unit cell by one step in the
    x and y direction on the twisted torus with lattice vectors
    a1 = (0, alpha) and a2 = (beta, gamma).

    Cells are numbered i = x0 + beta*y0 for 0 <= x0 < beta, 0 <= y0 < alpha.
    A y-step wraps modulo alpha.  An x-step from the last column (x0 = beta-1)
    returns to column 0 and shifts the row by -gamma, which implements the
    identification x^beta y^gamma = 1.  With gamma = 0 this is the ordinary
    (untwisted) alpha x beta torus.
    """
    n = alpha * beta
    X = matrix(Fq, n, n, 0)
    Y = matrix(Fq, n, n, 0)
    gamma_mod = gamma % alpha

    for y0 in range(alpha):
        for x0 in range(beta):
            i = x0 + beta * y0

            # y-shift
            x2 = x0
            y2 = (y0 + 1) % alpha
            j  = x2 + beta * y2
            Y[i, j] = Fq.one()

            # x-shift with twist
            if x0 < beta - 1:
                x2 = x0 + 1
                y2 = y0
            else:
                x2 = 0
                y2 = (y0 - gamma_mod) % alpha
            j = x2 + beta * y2
            X[i, j] = Fq.one()

    return X, Y

##############################################
# Evaluate Laurent polynomial on X,Y
##############################################

def poly_matrix(F, X, Y):
    """The matrix F(X, Y): each monomial c x^a y^b of F contributes
    c * X^a * Y^b (negative exponents use the inverse matrices)."""
    n = X.nrows()
    M = matrix(Fq, n, n, 0)
    for (ax, ay), coeff in F.dict().items():
        Xpow = X**ax  if ax >= 0 else (X**(-ax)).inverse()
        Ypow = Y**ay  if ay >= 0 else (Y**(-ay)).inverse()
        M += coeff * Xpow * Ypow
    return M

##############################################
# CSS check (twisted-toric: Av=(f,g), Bp=(ḡ,f̄))
##############################################

def is_css_for_geometry(F, G, alpha, beta, gamma):
    """
    True if (F, G) defines a valid weight-6 CSS code on the given torus.

    The check matrices are H_X = [f(X,Y) | -g(X,Y)] and
    H_Z = [gbar(X,Y) | fbar(X,Y)]; the relative minus sign is required for
    odd q (Lemma 1 of the paper) and may sit in either block.  Two tests:
      * every row of H_X and H_Z must have exactly wt(F) + wt(G) nonzero
        entries; a smaller weight means that two monomials of F or of G land
        on the same qudit because of the periodic identifications, and such
        tori are rejected;
      * H_X * H_Z^T = 0, i.e. all X-checks commute with all Z-checks.
    """
    X, Y = twisted_shifts(alpha, beta, gamma)

    Mf = poly_matrix(F, X, Y)    # f(X,Y)
    Mg = poly_matrix(G, X, Y)    # g(X,Y)

    Fbar = antipode(F)
    Gbar = antipode(G)
    Mf_bar = poly_matrix(Fbar, X, Y)   # \bar f(X,Y)
    Mg_bar = poly_matrix(Gbar, X, Y)   # \bar g(X,Y)

    # Compress zeros in Eq. (3): Hx=[f g], Hz=[-ḡ f̄]
    Hx = Mf.augment(-Mg)
    Hz = Mg_bar.augment(Mf_bar)

    # reject if monomials of F (or of G) wrap onto each other on the finite torus
    w = len(F.monomials()) + len(G.monomials())
    if any(len(r.nonzero_positions()) != w for r in Hx.rows()): return False
    if any(len(r.nonzero_positions()) != w for r in Hz.rows()): return False

    C  = Hx * Hz.transpose()
    try:
        return C.is_zero()
    except AttributeError:
        return C == 0

##############################################
# logical k(F,G,alpha,beta,gamma)
##############################################

def logical_k_FG(F, G, alpha, beta, gamma):
    """
    Number of logical qudits of the code (F, G) on the torus
    (alpha, beta, gamma), i.e. Eq. (k) of the paper:
    twice the dimension of R_q / <F, G, y^alpha - 1, x^beta y^gamma - 1>.
    gamma is reduced modulo alpha, which describes the same torus.
    Returns 0 if the Groebner computation fails.
    """
    gamma_mod = gamma % alpha
    gens = [
        F,
        G,
        y**alpha - 1,
        x**beta * y**gamma_mod - 1
    ]
    try:
        _, basis = laurent_groebner_and_basis(P, gens, order='lex')
    except (TypeError, ValueError, RuntimeError):
        return 0
    return 2 * len(basis)

##############################################
# Search over (alpha,beta,gamma)
##############################################

def search_alpha_beta_k_improve_FG(F, G,
                                   alpha_min, alpha_max,
                                   beta_min, beta_max,
                                   gamma_min , gamma_max, 
                                   k_target,
                                   n_target=None):
    """
    For a fixed pair (F, G), run over all tori (alpha, beta, gamma) with
    2*alpha*beta = n_target and -alpha < gamma < alpha (within the given
    bounds) and collect those on which the code has exactly k_target
    logical qudits.

    For each (alpha, beta) the untwisted torus (gamma = 0) is evaluated
    first: its k is recorded as k0, and pairs whose untwisted k already
    exceeds the target are skipped.  Both the untwisted and the twisted
    torus must pass the CSS check.

    Returns a list of tuples (alpha, beta, gamma, k0, k_gamma).
    """
    rows = []
    seen = set()

    for alpha in range(alpha_min, alpha_max + 1):
        for beta in range(beta_min, beta_max + 1):

            if n_target is not None and 2 * alpha * beta != n_target:
                continue

            k0   = logical_k_FG(F, G, alpha, beta, 0)
            if k0 > k_target:
                continue 
            css0 = is_css_for_geometry(F, G, alpha, beta, 0)
            
            g_lo = max(gamma_min, -alpha + 1)
            g_hi = min(gamma_max,  alpha - 1)

            for gamma in range(g_lo, g_hi + 1):
                k_gamma = logical_k_FG(F, G, alpha, beta, gamma)

                if k_gamma != k_target:
                    continue

                cssg = is_css_for_geometry(F, G, alpha, beta, gamma)

                if css0 and cssg:
                    rows.append((alpha, beta, gamma, k0, k_gamma))

    return rows

##############################################
# Ansatz
##############################################

def random_fg_eq5(max_abs_exp):
    """
    Draw one random pair (f, g) of the weight-6 ansatz of the paper,

        f = 1 + l1 x^a1 y^b1 + l2 x^a2 y^b2,
        g = 1 + m1 x^c1 y^d1 + m2 x^c2 y^d2,

    with exponents in [-max_abs_exp, max_abs_exp], all four monomials
    distinct and different from 1, and coefficients l_i, m_j chosen
    uniformly among the nonzero elements of F_q.
    """
    used = {(0, 0)}  # forbid the constant monomial
    def rand_exp():
        while True:
            a = random.randint(-max_abs_exp, max_abs_exp)
            b = random.randint(-max_abs_exp, max_abs_exp)
            if (a, b) not in used:
                used.add((a, b))
                return a, b

    a1, b1 = rand_exp()
    a2, b2 = rand_exp()
    c1, d1 = rand_exp()
    c2, d2 = rand_exp()

    def rand_coeff():
        while True:
            c = Fq.random_element()
            if c != 0:
                return c

    alpha1 = rand_coeff()
    alpha2 = rand_coeff()
    beta1  = rand_coeff()
    beta2  = rand_coeff()

    f = 1 + alpha1 * x**a1 * y**b1 + alpha2 * x**a2 * y**b2 
    g = 1 + beta1  * x**c1 * y**d1 + beta2  * x**c2 * y**d2 

    return f, g


##############################################
# Outer search over random (f,g)
##############################################

def search_random_fg(num_fg,
                     max_exp,
                     alpha_min, alpha_max,
                     beta_min, beta_max,
                     gamma_min, gamma_max, 
                     k_target,
                     out_filename,
                     n_target=None):
    """
    The search itself.  num_fg random pairs (f, g) are drawn; a pair is
    kept if it is coprime and if k_max = 2 dim R_q/<f, g> is at least
    k_target (a code can never have more logical qudits on a torus than
    k_max).  For each surviving pair all tori of block length n_target are
    examined with search_alpha_beta_k_improve_FG.  Every torus that gives
    k = k_target is written as one row of the CSV file out_filename, with
    columns f, g, n, alpha, beta, gamma, k_gamma0, k_gamma.

    Returns the number of rows written.
    """
    all_rows = []
    seen = set()

    for t in range(num_fg):
        f, g = random_fg_eq5(max_exp)

        if not laurent_coprime_fg(f, g):
            continue

        try:
            _, basis_plane = laurent_groebner_and_basis(P, [f, g], order='lex')
        except (TypeError, ValueError, RuntimeError):
            continue

        r = len(basis_plane)
        k_max = 2 * r
        if k_max < k_target:
            continue

        rows_fg = search_alpha_beta_k_improve_FG(
            f, g,
            alpha_min, alpha_max,
            beta_min, beta_max,
            gamma_min, gamma_max, 
            k_target,
            n_target=n_target
        )

        for (alpha, beta, gamma, k0, k_gamma) in rows_fg:
            key = (str(f), str(g), int(alpha), int(beta), int(gamma))
            if key in seen:
                continue 
            seen.add(key)
            all_rows.append((f, g, alpha, beta, gamma, k0, k_gamma))

    with open(out_filename, "w", newline="") as fcsv:
        writer = csv.writer(fcsv)
        writer.writerow(["f", "g", "n", "alpha", "beta", "gamma", "k_gamma0", "k_gamma"])
        for (f_poly, g_poly, alpha, beta, gamma, k0, kg) in all_rows:
            n_val = 2 * alpha * beta
            writer.writerow([str(f_poly),
                             str(g_poly),
                             int(n_val), 
                             int(alpha), int(beta), int(gamma),
                             int(k0), int(kg)])

    return len(all_rows)

##############################################
# eliminate minus: helpers
#
# The GAP distance script works with ordinary polynomials, so f and g are
# rewritten without negative exponents.  Multiplying a polynomial by a
# monomial x^sx y^sy only translates the check pattern on the lattice and
# defines the same code.  The helpers below parse the polynomial strings
# written by Sage, shift the exponents, and print them again.
##############################################

def clean_poly_string(s):
    """Keep only the characters x, y, digits, +, -, ^, * and spaces."""
    if s is None:
        return ""
    s = str(s)
    allowed = set("xyXY0123456789+-^* ")
    return "".join(ch for ch in s if ch in allowed)

def split_terms(expr):
    """Split "x + x*y^-1 + 1" into ["x", "+x*y^-1", "+1"]; a sign directly
    after '^' belongs to an exponent and does not start a new term."""
    expr = expr.replace(' ', '')
    if not expr:
        return []
    terms = []
    current = ''
    for i, ch in enumerate(expr):
        if ch in '+-' and i > 0 and expr[i-1] != '^':
            terms.append(current)
            current = ch
        else:
            current += ch
    if current:
        terms.append(current)
    return terms

def parse_term(term):
    """Read one term such as -3*x^2*y^-1 and return (ax, ay, coeff)
    with term = coeff * x^ax * y^ay."""
    t = term.replace(' ', '')
    if not t:
        return None

    coeff = 1
    if t[0] == '-':
        coeff = -1
        t = t[1:]
    elif t[0] == '+':
        t = t[1:]

    factors = t.split('*') if t else []

    if factors and re.fullmatch(r'-?\d+', factors[0]):
        coeff *= int(factors[0])
        factors = factors[1:]

    ax = 0
    ay = 0
    for f in factors:
        if not f:
            continue
        if f[0] in ('x', 'y'):
            var = f[0]
            exp = 1
            if len(f) > 1:
                if f[1] != '^':
                    raise ValueError(f"Bad factor {f} in term {term}")
                exp_str = f[2:]
                if exp_str == '':
                    raise ValueError(f"Missing exponent in {f}")
                exp = int(exp_str)
            if var == 'x':
                ax += exp
            else:
                ay += exp
        else:
            raise ValueError(f"Unknown factor {f} in term {term}")
    return ax, ay, coeff

def parse_poly(poly_str):
    """A polynomial string -> list of [ax, ay, coeff], one entry per term."""
    s = clean_poly_string(poly_str)
    terms = split_terms(s)
    return [list(parse_term(t)) for t in terms if t]

def shift_terms(terms):
    """Multiply the polynomial by x^sx y^sy so that all exponents become
    >= 0; returns the shifted terms and the shifts (sx, sy)."""
    if not terms:
        return [], 0, 0
    minx = min(a for a,_,_ in terms)
    miny = min(b for _,b,_ in terms)
    sx = -minx if minx < 0 else 0
    sy = -miny if miny < 0 else 0
    new_terms = [[a+sx, b+sy, c] for a,b,c in terms]
    return new_terms, sx, sy

def terms_to_string(terms):
    """Inverse of parse_poly for nonnegative exponents, e.g.
    [[2,1,1],[0,5,3],[0,0,-1]] -> "x^2*y+3*y^5-1"."""
    pieces = []
    for idx, (ax, ay, c) in enumerate(terms):
        if c == 0:
            continue
        sign = '-' if c < 0 else '+'
        abs_c = -c if c < 0 else c

        mon_parts = []
        if ax != 0:
            mon_parts.append('x' if ax == 1 else f'x^{ax}')
        if ay != 0:
            mon_parts.append('y' if ay == 1 else f'y^{ay}')

        if mon_parts:
            if abs_c == 1:
                body = '*'.join(mon_parts)
            else:
                body = f'{abs_c}*' + '*'.join(mon_parts)
        else:
            body = str(abs_c)

        if idx == 0:
            pieces.append(body if sign == '+' else '-' + body)
        else:
            pieces.append(sign + body)

    return ''.join(pieces) if pieces else '0'

def eliminate_minus(input_csv, output_csv):
    """
    Read the search output (input_csv) and write output_csv with eight
    extra columns: the cleaned strings f_clean, g_clean, the shifts
    f_shift_x, f_shift_y, g_shift_x, g_shift_y, and the shifted polynomials
    f_no_neg, g_no_neg without negative exponents.  The output is the input
    of the GAP distance script.
    """
    with open(input_csv, newline='') as fin, open(output_csv, 'w', newline='') as fout:
        reader = csv.DictReader(fin)

        fieldnames = reader.fieldnames + [
            "f_clean", "g_clean",
            "f_shift_x", "f_shift_y",
            "g_shift_x", "g_shift_y",
            "f_no_neg", "g_no_neg",
        ]
        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        writer.writeheader()

        for row_index, row in enumerate(reader, start=1):
            f_raw = row.get("f", "")
            g_raw = row.get("g", "")

            f_clean = clean_poly_string(f_raw)
            g_clean = clean_poly_string(g_raw)

            f_terms = parse_poly(f_clean)
            g_terms = parse_poly(g_clean)

            f_shifted, sx_f, sy_f = shift_terms(f_terms)
            g_shifted, sx_g, sy_g = shift_terms(g_terms)

            row["f_clean"]   = f_clean
            row["g_clean"]   = g_clean
            row["f_shift_x"] = sx_f
            row["f_shift_y"] = sy_f
            row["g_shift_x"] = sx_g
            row["g_shift_y"] = sy_g
            row["f_no_neg"]  = terms_to_string(f_shifted)
            row["g_no_neg"]  = terms_to_string(g_shifted)

            writer.writerow(row)

# Part 2
##############################################
# One complete run (search + eliminate minus) for a given n_target
##############################################

@parallel(ncpus=4)          # <-- number of simultaneous runs, set to your core count
def run_for_n(n_target):
    """
    Complete run for one block length n_target: set the random seed, choose
    the ranges of alpha, beta and gamma (all factorizations 2*alpha*beta =
    n_target and all twists are tried), run the search, and produce the two
    CSV files in data/data_{n}n_{k}k_{q}q_{w}w/.  Sage's @parallel runs this
    function for several n at the same time, each in its own process.
    """
    # same seed in every run, as before
    if seed is not None:
        random.seed(int(seed))
        try:
            from sage.misc.randstate import set_random_seed
            set_random_seed(int(seed))
        except ImportError:
            pass

    alpha_min, alpha_max = 1, n_target // 2 
    beta_min,  beta_max  = 1, n_target // 2
    gamma_min, gamma_max = -n_target // 2, n_target // 2   # covers every twist, since |gamma| < alpha <= n/2

    data_dir = f"data/data_{n_target}n_{k_target}k_{q}q_{w}w"
    os.makedirs(data_dir, exist_ok=True)

    out_filename = f"{data_dir}/n{n_target}fg_{k_target}k_{q}q_{w}wait.csv"
    minus_filename = f"{data_dir}/minus_n{n_target}fg_{k_target}k_{q}q_{w}wait.csv"

    nrows = search_random_fg(num_fg,
                             max_exp,
                             alpha_min, alpha_max,
                             beta_min, beta_max,
                             gamma_min, gamma_max, 
                             k_target,
                             out_filename=out_filename,
                             n_target=n_target)

    eliminate_minus(out_filename, minus_filename)

    return f"n={n_target}: saved {nrows} CSS rows to {out_filename} and {minus_filename}"

# Part 3
# Block lengths n (number of physical qudits) to search; each n runs in its own process.
n_list = [42]

for (args, kwargs), result in run_for_n(n_list):
    print(result)

n=42: saved 146 CSS rows to data/data_42n_4k_3q_6w/n42fg_4k_3q_6wait.csv and data/data_42n_4k_3q_6w/minus_n42fg_4k_3q_6wait.csv
